In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from  lora_transfer_pruning.core.pruning_instrumentor import PruningInstrumentor
from transformers import AutoModelForCausalLM
import torch
import compare_utils
from compare_utils import debug_group_prune_step_by_step
from lora_transfer_pruning.adapter.torch_pruning_group_builder import TorchPruningGroupBuilder
from compare_utils import full_attention_test_with_prune

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
MODEL = "google/gemma-4-E4B-it" 
DEVICE = "cuda:3"
def load_model(device=DEVICE):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL,
        #quantization_config=quantization_config,
        #dtype=torch.bfloat16,
        device_map=device,
        # cache_dir="/glazkov-dev/.cache",
    )
    return model

In [3]:
model = load_model()

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

In [4]:
from transformer_lens.model_bridge import TransformerBridge
import transformer_lens

bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model,
    dtype=torch.float16,
)

In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
validation_dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="validation",
)
validation_dataset

Dataset({
    features: ['text'],
    num_rows: 3760
})

In [6]:
for i, text in enumerate(validation_dataset):
    print(f"{i}: {text}")
    if i > 10:
        break

0: {'text': ''}
1: {'text': ' = Homarus gammarus = \n'}
2: {'text': ''}
3: {'text': ' Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , producing eggs which are carried by the females for up to a year before hatching into planktonic larvae . Homarus gammarus is a highly esteemed food , and is widely caught using lobster pots , mostly around the British Isles . \n'}
4: {'text': ''}
5: {'text': ' = = Description = = \n'}
6: {'text': ''}
7: {'text': ' Homarus gammarus is a large crustacean , with a body length up to 60 centimetres ( 24 in ) and weighing up to 5 – 6 kilogram

In [7]:
CONTEXT_LENGTH = 256
NUM_EVAL_BLOCKS = 32 #(block=batch)
EVAL_BATCH_SIZE = 1

validation_text = "\n\n".join(
    text for text in validation_dataset["text"] if text.strip()
)
validation_tokens = tokenizer(
    validation_text,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids[0]

num_blocks = NUM_EVAL_BLOCKS
assert num_blocks > 0, "Validation split does not contain enough tokens"
evaluation_blocks = validation_tokens[: num_blocks * CONTEXT_LENGTH].reshape(
    num_blocks, CONTEXT_LENGTH
)
evaluation_blocks.shape

torch.Size([32, 256])

In [8]:
# Fraction-based version of the activation-vs-structural comparison.
# Run on a freshly loaded, unpruned bridge; skip the preceding explicit-index
# comparison cell after restarting the kernel.
from compare_utils import compare_tp_and_transfer_pruning
from compare_utils import create_prune_task

FRACTION_ATTN_LAYERS = [0]
FRACTION_MLP_LAYERS = [4, 12, 20]
#ATTN_OUT_FRACTION = 0.1
#MLP_OUT_FRACTION = 0.3 
ATTN_OUT_FRACTION = [1, 2, 99]
MLP_OUT_FRACTION = [3, 5, 1000]
FRACTION_SEED = 0

prune_task = create_prune_task(FRACTION_ATTN_LAYERS, FRACTION_MLP_LAYERS, ATTN_OUT_FRACTION, MLP_OUT_FRACTION)

In [9]:
import importlib
importlib.reload(compare_utils)

<module 'compare_utils' from '/glazkov-dev/LoRa-Transfer-Pruning/experiments/compare_tp_and_our/compare_utils.py'>

In [10]:
first_divergence, comparison_rows = full_attention_test_with_prune(bridge, 
                                                                   0, 
                                                                   evaluation_blocks, 
                                                                   prune_task,
                                                                   record_activations=True)

/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.language_model.layers.24._original_component.pre_feedforward_layernorm._original_component.weight', 'model.language_model.layers.29._original_component.input_layernorm._original_component.weight', 'model.audio_tower.layers.6.self_attn.post.linear.weight', 'model.language_model.layers.4._original_component.self_attn._original_component.k_norm._original_component.weight', 'model.language_model.layers.8._original_component.post_feedforward_layernorm._original_component.weight', 'model.language_model.layers.29._original_component.post_attention_layernorm._original_component.weight', 'model.language_model.layers.32._original_component.post_attention_layernorm._original_component.weight', 'model.vision_tower._original_component.encoder.layers.7.self_attn.q_norm.weight', 'model.vision_tower._original_component.encoder.layers.12.self_a

[00] hook_in                          OK   shape=(1, 1, 2560) max=0.000e+00 mean=0.000e+00 rmse=0.000e+00 rel_l2=0.000e+00 abs_l2=0.000e+00
[01] _original_component.q_proj.hook_in OK   shape=(1, 1, 2560) max=0.000e+00 mean=0.000e+00 rmse=0.000e+00 rel_l2=0.000e+00 abs_l2=0.000e+00
[02] _original_component.q_proj.hook_out OK   shape=(1, 1, 2000) max=0.000e+00 mean=0.000e+00 rmse=0.000e+00 rel_l2=0.000e+00 abs_l2=0.000e+00 aligned=q_flat_idxs, dim=2
[03] _original_component.q_norm.hook_in OK   shape=(1, 1, 8, 250) max=0.000e+00 mean=0.000e+00 rmse=0.000e+00 rel_l2=0.000e+00 abs_l2=0.000e+00 aligned=local_head_idxs, dim=3
[04] _original_component.q_norm.hook_out DIFF shape=(1, 1, 8, 250) max=3.125e-02 mean=9.540e-04 rmse=2.768e-03 rel_l2=2.812e-03 abs_l2=1.238e-01 aligned=local_head_idxs, dim=3
[05] _original_component.k_proj.hook_in OK   shape=(1, 1, 2560) max=0.000e+00 mean=0.000e+00 rmse=0.000e+00 rel_l2=0.000e+00 abs_l2=0.000e+00
[06] _original_component.k_proj.hook_out OK   shape=(1,

In [11]:
comparison_rows[3]

{'stage': '_original_component.q_norm.hook_in',
 'close': True,
 'reference_shape': (1, 1, 8, 250),
 'candidate_shape': (1, 1, 8, 256),
 'alignment': 'local_head_idxs, dim=3',
 'max_abs': 0.0,
 'mean_abs': 0.0,
 'rmse': 0.0,
 'rel_l2': 0.0,
 'abs_l2': 0.0,
 'reference_activation': tensor([[[[-204.0000,  105.5000,  -27.0000,  ...,    9.9375,  -26.1250,
             -99.5000],
           [-114.0000,   84.5000,   32.2500,  ...,   30.0000,   -6.2500,
             -57.2500],
           [ -17.6250,   36.5000,  -20.1250,  ...,   82.0000,  -58.2500,
             -25.6250],
           ...,
           [  35.5000,   57.2500, -114.5000,  ...,  -16.8750,   61.0000,
              23.8750],
           [  18.7500,  -19.2500,   -1.1719,  ...,   62.5000,   25.1250,
             -63.5000],
           [ 101.0000,  126.5000,  -14.3750,  ...,    2.9375,   11.3750,
             -26.3750]]]]),
 'candidate_activation': tensor([[[[-204.0000,  105.5000,  -27.0000,  ...,    9.9375,  -26.1250,
             -99.500

In [12]:
comparison_rows[4]

{'stage': '_original_component.q_norm.hook_out',
 'close': False,
 'reference_shape': (1, 1, 8, 250),
 'candidate_shape': (1, 1, 8, 256),
 'alignment': 'local_head_idxs, dim=3',
 'max_abs': 0.03125,
 'mean_abs': 0.000953960872720927,
 'rmse': 0.002767580095678568,
 'rel_l2': 0.002811573663294597,
 'abs_l2': 0.12376994639635086,
 'reference_activation': tensor([[[[-4.5938,  2.3750, -0.6094,  ...,  0.2246, -0.5898, -2.2500],
           [-3.7812,  2.7969,  1.0703,  ...,  0.9961, -0.2070, -1.8984],
           [-0.5352,  1.1094, -0.6133,  ...,  2.5000, -1.7734, -0.7812],
           ...,
           [ 0.6211,  1.0000, -2.0000,  ..., -0.2949,  1.0625,  0.4160],
           [ 0.3457, -0.3555, -0.0216,  ...,  1.1562,  0.4648, -1.1719],
           [ 2.1094,  2.6406, -0.2988,  ...,  0.0613,  0.2373, -0.5508]]]]),
 'candidate_activation': tensor([[[[-4.5938,  2.3750, -0.6094,  ...,  0.2236, -0.5898, -2.2344],
           [-3.7812,  2.8125,  1.0703,  ...,  0.9961, -0.2070, -1.8984],
           [-0.535

In [13]:
bridge.blocks[0].attn.q_norm.hook_in

HookPoint(name='blocks.0.attn.q_norm.hook_in')

In [14]:
comparison_rows[4]['reference_activation'].shape

torch.Size([1, 1, 8, 250])

In [15]:
coef = ((250 / 256) ** 0.5)

In [16]:
new_t = comparison_rows[4]['candidate_activation'].clone() * coef

In [17]:
comparison_rows[4]['candidate_activation']

tensor([[[[-4.5938,  2.3750, -0.6094,  ...,  0.2236, -0.5898, -2.2344],
          [-3.7812,  2.8125,  1.0703,  ...,  0.9961, -0.2070, -1.8984],
          [-0.5352,  1.1094, -0.6133,  ...,  2.5000, -1.7734, -0.7812],
          ...,
          [ 0.6211,  1.0000, -2.0000,  ..., -0.2949,  1.0625,  0.4160],
          [ 0.3457, -0.3555, -0.0216,  ...,  1.1484,  0.4629, -1.1719],
          [ 2.0938,  2.6406, -0.2988,  ...,  0.0613,  0.2373, -0.5469]]]])

In [18]:
device = bridge.blocks[0].attn.q_norm._original_component.weight.device

In [19]:
bridge.blocks[0].attn.q_norm._original_component.weight.dtype

torch.bfloat16

In [20]:
comparison_rows[3]['candidate_activation'].dtype

torch.float32

In [21]:
ans_f32 = torch.pow(comparison_rows[3]['reference_activation'].to(device).pow(2).mean(-1, keepdim=True) + 1e-6, -0.5) \
    * comparison_rows[3]['reference_activation'].to(device) * bridge.blocks[0].attn.q_norm._original_component.weight
    
ans_bf = ans_f32.to(bridge.blocks[0].attn.q_norm._original_component.weight.dtype)
ans_bf


tensor([[[[-4.5938,  2.3750, -0.6094,  ...,  0.2246, -0.5898, -2.2500],
          [-3.7812,  2.7969,  1.0703,  ...,  0.9961, -0.2070, -1.8984],
          [-0.5352,  1.1094, -0.6133,  ...,  2.5000, -1.7734, -0.7812],
          ...,
          [ 0.6211,  1.0000, -2.0000,  ..., -0.2949,  1.0625,  0.4160],
          [ 0.3457, -0.3555, -0.0216,  ...,  1.1562,  0.4648, -1.1719],
          [ 2.1094,  2.6406, -0.2988,  ...,  0.0613,  0.2373, -0.5508]]]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<ToCopyBackward0>)

In [22]:
import torch.nn.functional as F

In [23]:
x = torch.randn(2, 3, 4)
x_new = F.pad(x, (0, 6))
x_new.shape

torch.Size([2, 3, 10])

In [24]:
aug_ref = F.pad(comparison_rows[3]['reference_activation'], (0, 6))
aug_weight = F.pad(bridge.blocks[0].attn.q_norm._original_component.weight, (0, 6))

In [45]:
torch.allclose(comparison_rows[3]['reference_activation'], comparison_rows[3]['candidate_activation'])
print(comparison_rows[3]['reference_activation'].dtype)
print(comparison_rows[3]['candidate_activation'].dtype)

torch.float32
torch.float32


In [25]:
aug_ref.dtype, aug_weight.float().dtype

(torch.float32, torch.float32)

In [26]:
aug_ref.shape

torch.Size([1, 1, 8, 256])

In [27]:
ans_f32_2 = torch.pow(aug_ref.to(device).pow(2).mean(-1, keepdim=True) + 1e-6, -0.5) * aug_ref.to(device) * aug_weight.float()
    
# ans_bf_2 = ans_f32_2.to(bridge.blocks[0].attn.q_norm._original_component.weight.dtype)
# ans_bf_2

In [28]:
coef = ((torch.tensor(250) / 256)).sqrt()

In [29]:
reference_check = (ans_f32_2* coef)[:, :, :, :-6] #needs to end on 2.2500
reference_check #is strange, matematically it should be correct.

tensor([[[[-4.6044,  2.3812, -0.6094,  ...,  0.2243, -0.5897, -2.2458],
          [-3.7822,  2.8034,  1.0700,  ...,  0.9953, -0.2074, -1.8994],
          [-0.5368,  1.1116, -0.6129,  ...,  2.4974, -1.7741, -0.7804],
          ...,
          [ 0.6200,  0.9999, -1.9998,  ..., -0.2947,  1.0654,  0.4170],
          [ 0.3462, -0.3554, -0.0216,  ...,  1.1540,  0.4639, -1.1725],
          [ 2.1051,  2.6366, -0.2996,  ...,  0.0612,  0.2371, -0.5497]]]],
       device='cuda:3', grad_fn=<SliceBackward0>)

In [33]:
bfloat = aug_weight.dtype

In [30]:
reference_check.to(aug_weight.dtype) #совпало!

tensor([[[[-4.5938,  2.3750, -0.6094,  ...,  0.2246, -0.5898, -2.2500],
          [-3.7812,  2.7969,  1.0703,  ...,  0.9961, -0.2070, -1.8984],
          [-0.5352,  1.1094, -0.6133,  ...,  2.5000, -1.7734, -0.7812],
          ...,
          [ 0.6211,  1.0000, -2.0000,  ..., -0.2949,  1.0625,  0.4160],
          [ 0.3457, -0.3555, -0.0216,  ...,  1.1562,  0.4648, -1.1719],
          [ 2.1094,  2.6406, -0.2988,  ...,  0.0613,  0.2373, -0.5508]]]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<ToCopyBackward0>)

In [31]:
#get candidate values, but need reference!
torch.allclose(reference_check.cpu(), comparison_rows[4]['candidate_activation'].to(reference_check.dtype))

False

In [32]:
comparison_rows[4]['reference_activation']

tensor([[[[-4.5938,  2.3750, -0.6094,  ...,  0.2246, -0.5898, -2.2500],
          [-3.7812,  2.7969,  1.0703,  ...,  0.9961, -0.2070, -1.8984],
          [-0.5352,  1.1094, -0.6133,  ...,  2.5000, -1.7734, -0.7812],
          ...,
          [ 0.6211,  1.0000, -2.0000,  ..., -0.2949,  1.0625,  0.4160],
          [ 0.3457, -0.3555, -0.0216,  ...,  1.1562,  0.4648, -1.1719],
          [ 2.1094,  2.6406, -0.2988,  ...,  0.0613,  0.2373, -0.5508]]]])

Converting from fp32 to bf have big difference!

In [99]:
aug_ref = aug_ref.to(bfloat)
aug_weight = aug_weight.to(bfloat)

coef = ((250/256) ** 0.5 ) * 1
# aug_ref_input = aug_ref.float() * coef
# aug_ref_input = aug_ref_input.to(bfloat)
aug_ref_input = aug_ref

ans_bf = torch.pow(aug_ref_input.to(device).float().pow(2).mean(-1, keepdim=True) + 1e-6, -0.5) * aug_ref_input.to(device).float() * aug_weight.float()
ans_bf = ans_bf.to(bfloat)
# ans_bf = ans_bf.float() #doesnt help
# coef = (250/256) ** 0.5 #multiplying in bloat is inaccurate, we need to multiply in fp32
check_ans = (ans_bf[:, :, :, :-6] * coef)
check_ans = check_ans.to(bfloat)
check_ans

tensor([[[[-4.5938,  2.3750, -0.6094,  ...,  0.2236, -0.5898, -2.2344],
          [-3.7812,  2.8125,  1.0703,  ...,  0.9961, -0.2070, -1.8984],
          [-0.5352,  1.1094, -0.6133,  ...,  2.5000, -1.7734, -0.7812],
          ...,
          [ 0.6211,  1.0000, -2.0000,  ..., -0.2949,  1.0625,  0.4160],
          [ 0.3457, -0.3555, -0.0216,  ...,  1.1484,  0.4629, -1.1719],
          [ 2.0938,  2.6406, -0.2988,  ...,  0.0613,  0.2373, -0.5469]]]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<MulBackward0>)

In [65]:
torch.allclose(check_ans.cpu().float(), comparison_rows[4]['reference_activation'])

False

There is problem of applying coefficient after RMSNorm calculation, because in Llama and Gemma returned value casts to input_dtype=bfloat. And bfloat * coef has bad precision.

Probable solutions: change forward, custom rmsnorm in hook instead of forward, change shape of input tensor dropping zeroes (will work perfectly, but needs additional memory on activation).